In [2]:
import pandas as pd
import numpy as np
import re
import lightgbm as lgb

In [17]:
DATA_FOLDER = "./"

df = pd.read_pickle(DATA_FOLDER + "df_fe_epic_big_best_customers_25c.pickle")
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
product_ids = pd.read_csv(DATA_FOLDER + "product_id_apredecir201912.txt", sep="\t")[
    "product_id"
].tolist()

In [18]:

# Transformar object a category (menos claves)
columns = df.select_dtypes(include=["object"]).columns.tolist()
for col in columns:
    if col not in ["product_id", "customer_id", "date_id"]:
        df[col] = df[col].astype("category")

# Eliminar columnas datetime innecesarias
datetime_cols = df.select_dtypes(include=["datetime"]).columns.tolist()
for col in datetime_cols:
    if col != "date_id":
        df.drop(columns=[col], inplace=True)


In [19]:
# creo el target
kaggle_df = df[df["date_id"] == df["date_id"].max()].copy()
kaggle_df = kaggle_df[kaggle_df["product_id"].isin(product_ids)].copy()

df["target"] = df.groupby(["product_id", "customer_id"])["tn"].shift(-2)
# elimino las rows donde target es nan
df = df[df["target"].notna()]

# entreno hasta date_id = 33 incluido
final_train_index = df[df["date_id"] <= 33].index
# testeo en date_id = 33
test_index = df[df["date_id"] == 33].index
# entreno hasta date_id 31 incluido
train_index = df[df["date_id"] <= 31].index


/tmp/ipykernel_325102/754139255.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["target"] = df.groupby(["product_id", "customer_id"])["tn"].shift(-2)


In [ ]:
# Reemplazo de inf por nan
df.replace([np.inf, -np.inf], np.nan, inplace=True)



# Crear splitter con 20% y 2 meses

# Columnas a dropear
drop_cols = ["fecha", "target", "date_id"]

In [26]:
train_data = lgb.Dataset(
    df.loc[train_index].drop(columns=drop_cols),
    label=df.loc[train_index, "target"],
    categorical_feature="auto",
)
test_data = lgb.Dataset(
    df.loc[test_index].drop(columns=drop_cols),
    label=df.loc[test_index, "target"],
    categorical_feature="auto",
    reference=train_data,
)

model = lgb.train(
    params={
        "objective": "tweedie",
        "metric": "rmse",
        "boosting_type": "gbdt",
        "num_leaves": 31,
        "learning_rate": 0.05,
        "feature_fraction": 0.5,
        "bagging_fraction": 0.8,
        "bagging_freq": 5,
        "verbose": -1,
        #"device": "cuda",
    },
    train_set=train_data,
    valid_sets=[test_data],
    num_boost_round=9999,
    callbacks=[
        lgb.early_stopping(int(400 + 4 / 0.05), verbose=True),
        lgb.log_evaluation(period=10),
    ],
)


Training until validation scores don't improve for 480 rounds
[10]	valid_0's rmse: 7.07792
[20]	valid_0's rmse: 6.69191
[30]	valid_0's rmse: 6.12621
[40]	valid_0's rmse: 5.39546
[50]	valid_0's rmse: 4.70102
[60]	valid_0's rmse: 4.19237
[70]	valid_0's rmse: 3.94923
[80]	valid_0's rmse: 3.85693
[90]	valid_0's rmse: 3.81863
[100]	valid_0's rmse: 3.77773
[110]	valid_0's rmse: 3.74736
[120]	valid_0's rmse: 3.74403
[130]	valid_0's rmse: 3.73774
[140]	valid_0's rmse: 3.72489
[150]	valid_0's rmse: 3.71383
[160]	valid_0's rmse: 3.70237
[170]	valid_0's rmse: 3.67656
[180]	valid_0's rmse: 3.66857
[190]	valid_0's rmse: 3.65761
[200]	valid_0's rmse: 3.61945
[210]	valid_0's rmse: 3.63205
[220]	valid_0's rmse: 3.61827
[230]	valid_0's rmse: 3.58476
[240]	valid_0's rmse: 3.56494
[250]	valid_0's rmse: 3.56381
[260]	valid_0's rmse: 3.56322
[270]	valid_0's rmse: 3.55955
[280]	valid_0's rmse: 3.54195
[290]	valid_0's rmse: 3.54657
[300]	valid_0's rmse: 3.5464
[310]	valid_0's rmse: 3.53646
[320]	valid_0's rm

In [27]:
best_iteration = model.best_iteration
print(f"Best iteration: {best_iteration}")
predictions = model.predict(
    df.loc[test_index].drop(columns=drop_cols),
    num_iteration=best_iteration,
)
predictions = np.clip(predictions, 0, None)  # Asegurar que las predicciones no sean negativas
test_df = df.loc[test_index].copy()
test_df["predictions"] = predictions
test_df = test_df[["product_id", "customer_id", "date_id", "predictions", "target"]]
test_df = test_df[test_df["product_id"].isin(product_ids)]
test_df = test_df.groupby("product_id").agg(
    {
        "predictions": "sum",
        "target": "sum",
    }
)
test_df["abs_error"] = np.abs(test_df["predictions"] - test_df["target"])
total_error = np.sum(np.abs(test_df["predictions"] - test_df["target"])) / np.sum(test_df["target"])
print(f"Total error: {total_error:.4f}")
test_df

Best iteration: 936
Total error: 0.2330


,predictions,target,abs_error
product_id,,,
20001,1323.751414,1504.688599,180.937184
20002,976.793300,1087.308594,110.515294
20003,810.166671,892.501282,82.334611
20004,559.274298,637.900024,78.625727
20005,548.969241,593.244446,44.275205
...,...,...,...
21263,0.022110,0.012700,0.009410
21265,0.067261,0.050070,0.017191
21266,0.069598,0.051210,0.018388


In [29]:
# reentreno el modleo final para kaggle
final_train_data = lgb.Dataset(
    df.loc[final_train_index].drop(columns=drop_cols),
    label=df.loc[final_train_index, "target"],
    categorical_feature="auto",
)
final_model = lgb.train(
    params={
        "objective": "tweedie",
        "metric": "rmse",
        "boosting_type": "gbdt",
        "num_leaves": 31,
        "learning_rate": 0.05,
        "feature_fraction": 0.5,
        "bagging_fraction": 0.8,
        "bagging_freq": 5,
        "verbose": -1,
    },
    train_set=final_train_data,
    num_boost_round=best_iteration,
)

In [30]:
df.loc[final_train_index]

,product_id,customer_id,fecha,plan_precios_cuidados,cust_request_qty,cust_request_tn,tn,stock_final,cat1,cat2,...,prod_tn_rolling_mean_24_lag_1_x_tn_wavelet_0_max_lag_15,prod_tn_rolling_mean_24_lag_1_x_tn_wavelet_0_max,prod_tn_rolling_mean_24_lag_1_x_tn_wavelet_0_max_lag_2,prod_tn_wavelet_0_max_lag_11_x_tn_wavelet_0_max_lag_15,prod_tn_wavelet_0_max_lag_11_x_tn_wavelet_0_max,prod_tn_wavelet_0_max_lag_11_x_tn_wavelet_0_max_lag_2,prod_tn_wavelet_0_max_lag_15_x_tn_wavelet_0_max,prod_tn_wavelet_0_max_lag_15_x_tn_wavelet_0_max_lag_2,prod_tn_wavelet_0_max_x_tn_wavelet_0_max_lag_2,target
0,20001,0,2017-01,NaN,229,139.028275,139.028275,NaN,HC,ROPA LAVADO,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,170.502975
1,20001,0,2017-02,NaN,221,107.575981,106.344749,NaN,HC,ROPA LAVADO,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,74.377518
2,20001,0,2017-03,NaN,261,174.420517,170.502975,NaN,HC,ROPA LAVADO,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,348002.40625,291.073944
3,20001,0,2017-04,NaN,128,86.074211,74.377518,NaN,HC,ROPA LAVADO,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,348002.40625,271.754883
4,20001,0,2017-05,NaN,346,292.775238,291.073944,NaN,HC,ROPA LAVADO,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,348002.40625,134.584686
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
819323,21290,10021,2017-06,0.0,0,0.000000,0.000000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000
819326,21290,10022,2017-06,0.0,0,0.000000,0.000000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000
819329,21290,10023,2017-06,0.0,0,0.000000,0.000000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000
819332,21290,10024,2017-06,0.0,0,0.000000,0.000000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000


In [31]:
kaggle_df

,product_id,customer_id,fecha,plan_precios_cuidados,cust_request_qty,cust_request_tn,tn,stock_final,cat1,cat2,...,prod_tn_rolling_mean_24_lag_1_x_tn_wavelet_0_max_lag_11,prod_tn_rolling_mean_24_lag_1_x_tn_wavelet_0_max_lag_15,prod_tn_rolling_mean_24_lag_1_x_tn_wavelet_0_max,prod_tn_rolling_mean_24_lag_1_x_tn_wavelet_0_max_lag_2,prod_tn_wavelet_0_max_lag_11_x_tn_wavelet_0_max_lag_15,prod_tn_wavelet_0_max_lag_11_x_tn_wavelet_0_max,prod_tn_wavelet_0_max_lag_11_x_tn_wavelet_0_max_lag_2,prod_tn_wavelet_0_max_lag_15_x_tn_wavelet_0_max,prod_tn_wavelet_0_max_lag_15_x_tn_wavelet_0_max_lag_2,prod_tn_wavelet_0_max_x_tn_wavelet_0_max_lag_2
35,20001,0,2019-12,NaN,221,301.146118,295.231415,69.755478,HC,ROPA LAVADO,...,134285.031250,134285.031250,134285.031250,134285.031250,348002.406250,348002.406250,348002.406250,348002.406250,348002.406250,348002.406250
71,20001,10001,2019-12,0.0,18,214.721848,180.219376,69.755478,HC,ROPA LAVADO,...,119977.734375,119977.734375,119977.734375,119977.734375,364852.718750,364852.718750,364852.718750,364852.718750,364852.718750,364852.718750
107,20001,10002,2019-12,0.0,20,115.303223,113.331650,69.755478,HC,ROPA LAVADO,...,7417.522949,7417.522949,7417.522949,7417.522949,34536.617188,34536.617188,34536.617188,34536.617188,34536.617188,34536.617188
143,20001,10003,2019-12,0.0,9,113.981369,102.275169,69.755478,HC,ROPA LAVADO,...,37010.878906,37010.878906,37010.878906,37010.878906,89045.218750,89045.218750,89045.218750,89045.218750,89045.218750,89045.218750
179,20001,10004,2019-12,0.0,8,34.648102,34.648102,69.755478,HC,ROPA LAVADO,...,130266.921875,130266.921875,130266.921875,130266.921875,374149.343750,374149.343750,374149.343750,374149.343750,374149.343750,374149.343750
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
818517,21276,10021,2019-12,0.0,0,0.000000,0.000000,1.055920,PC,PIEL1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000
818527,21276,10022,2019-12,0.0,0,0.000000,0.000000,1.055920,PC,PIEL1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000
818537,21276,10023,2019-12,0.0,0,0.000000,0.000000,1.055920,PC,PIEL1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000
818547,21276,10024,2019-12,0.0,0,0.000000,0.000000,1.055920,PC,PIEL1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000


In [33]:
kaggle_predictions = final_model.predict(
    kaggle_df.drop(columns=drop_cols, errors="ignore"),)
kaggle_predictions = np.clip(kaggle_predictions, 0, None)  # Asegurar que las predicciones no sean negativas
submission_df = kaggle_df[["product_id", "customer_id"]].copy()
submission_df["predictions"] = kaggle_predictions
submission_df = submission_df[submission_df["product_id"].isin(product_ids)]
submission_df = submission_df.groupby("product_id").agg(
    {
        "predictions": "sum",
    }
)
submission_df.reset_index(inplace=True)
submission_df.rename(columns={"predictions": "tn"}, inplace=True)
submission_df.to_csv(DATA_FOLDER + "submission_lgb_no_scaling_simple.csv", index=False)
submission_df

,product_id,tn
0,20001,1125.061922
1,20002,940.968536
2,20003,716.318103
3,20004,452.346541
4,20005,379.054987
...,...,...
775,21263,0.016809
776,21265,0.064848
777,21266,0.073800
778,21267,0.074952
